# Tạo dữ liệu bài 1

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# --- CẤU HÌNH ---
num_rows = 1000000
num_stores = 50
num_products = 500
categories = ['Electronics', 'Grocery', 'Fashion', 'Home & Garden', 'Beauty']

# --- BƯỚC 1: TẠO SỰ PHÂN HÓA (Variance) ---

# 1.1. Tạo cửa hàng "lớn" và "nhỏ"
# Sử dụng phân bố lũy thừa ngược (Power Law) để tạo trọng số (probabilities)
store_probs = np.random.power(0.5, size=num_stores) # Càng nhỏ, sự phân hóa càng mạnh
store_probs = store_probs / store_probs.sum() # Đảm bảo tổng bằng 1

# 1.2. Tạo sản phẩm "hot" và sản phẩm "ế"
product_probs = np.random.power(0.3, size=num_products) 
product_probs = product_probs / product_probs.sum()

# 1.3. Tạo danh mục cho sản phẩm và gán giá trị đặc trưng
product_data = []
for i in range(1, num_products + 1):
    category = np.random.choice(categories, p=[0.1, 0.3, 0.3, 0.15, 0.15]) 
    
    # SỬA LỖI Ở ĐÂY: Dùng hàm round() bọc bên ngoài thay vì .round()
    if category in ['Electronics', 'Fashion']:
        price = round(np.random.gamma(shape=5, scale=50), 2) + 10 
    else:
        price = round(np.random.gamma(shape=2, scale=15), 2) + 5 
        
    product_data.append({'product_id': i, 'category': category, 'price': price})

product_df = pd.DataFrame(product_data)

# --- BƯỚC 2: SINH DỮ LIỆU GIAO DỊCH (1 triệu dòng) ---
data = {
    'transaction_id': np.arange(1, num_rows + 1),
    # Sử dụng `p=store_probs` để gán xác suất không đồng đều
    'store_id': np.random.choice(np.arange(1, num_stores + 1), size=num_rows, p=store_probs),
    # Sử dụng `p=product_probs` để gán xác suất không đồng đều
    'product_id': np.random.choice(np.arange(1, num_products + 1), size=num_rows, p=product_probs),
    'quantity': np.random.randint(1, 10, size=num_rows)
}

df = pd.DataFrame(data)

# Map category và price từ bảng product_df sang bảng df
df = df.merge(product_df, on='product_id', how='left')

# --- BƯỚC 3: TẠO THỜI GIAN VÀ XUẤT FILE ---
start_date = datetime(2025, 1, 1)
df['transaction_date'] = [start_date + timedelta(seconds=np.random.randint(0, 31536000)) 
                          for _ in range(num_rows)]

df = df.sort_values('transaction_date').reset_index(drop=True)

os.makedirs('../data', exist_ok=True)
# Giữ nguyên định dạng CSV theo yêu cầu bài trước
df.to_csv('../data/retail_transactions.csv', index=False)

print("Đã tạo lại file thành công tại ../data/retail_transactions_large.csv")
print("10 dòng đầu tiên của dữ liệu đã được cải tiến:")
print(df.head(10))
print(f"\nSchema của dữ liệu:")
print(df.info())

Đã tạo lại file thành công tại ../data/retail_transactions_large.csv
10 dòng đầu tiên của dữ liệu đã được cải tiến:
   transaction_id  store_id  product_id  quantity       category   price  \
0          620486         8          92         7        Grocery   20.94   
1          177622        27          72         3    Electronics  227.56   
2          923797        44         334         4        Fashion  212.43   
3          286596         3         162         4  Home & Garden   14.72   
4          224889        12         322         9         Beauty   46.33   
5          111718         2         403         7         Beauty   45.51   
6          300751        40         182         6  Home & Garden   86.80   
7          354852        23         328         5         Beauty   15.11   
8          857904        10         328         4         Beauty   15.11   
9          183221        10         259         4        Grocery   81.78   

     transaction_date  
0 2025-01-01 00:01:31  

# Tạo dữ liệu bài 2

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# --- CẤU HÌNH ---
num_rows = 1000000
num_users = 50000

# Thiết lập tỷ lệ (Xác suất) cho từng danh mục để tạo sự chênh lệch thực tế
pages = ['home', 'product_list', 'product_detail', 'cart', 'checkout', 'payment']
page_probs = [0.40, 0.25, 0.20, 0.08, 0.05, 0.02] # Phễu rơi rớt dần

actions = ['view', 'scroll', 'click', 'add_to_cart']
action_probs = [0.50, 0.25, 0.20, 0.05]

devices = ['mobile', 'desktop', 'tablet']
device_probs = [0.70, 0.25, 0.05] # Mobile chiếm ưu thế

countries = ['Vietnam', 'USA', 'Thailand', 'Singapore', 'Japan', 'Korea']
country_probs = [0.60, 0.15, 0.10, 0.05, 0.05, 0.05] # VN chiếm ưu thế

# Sinh trọng số cho người dùng (để có người truy cập nhiều, người ít)
user_probs = np.random.power(0.2, size=num_users)
user_probs = user_probs / user_probs.sum()
user_ids = [f"U{i:05d}" for i in range(1, num_users + 1)]

# --- SINH DỮ LIỆU ---
data = {
    'user_id': np.random.choice(user_ids, size=num_rows, p=user_probs),
    'page': np.random.choice(pages, size=num_rows, p=page_probs),
    'action': np.random.choice(actions, size=num_rows, p=action_probs),
    'device': np.random.choice(devices, size=num_rows, p=device_probs),
    'country': np.random.choice(countries, size=num_rows, p=country_probs)
}

df_pandas = pd.DataFrame(data)

# --- THÊM THỜI GIAN ---
start_date = datetime(2026, 1, 1)
df_pandas['timestamp'] = [start_date + timedelta(seconds=np.random.randint(0, 90*24*3600)) 
                          for _ in range(num_rows)]

df_pandas = df_pandas.sort_values('timestamp').reset_index(drop=True)

# Xuất file
os.makedirs('../data', exist_ok=True)
df_pandas.to_csv('../data/web_logs.csv', index=False)
print("Đã tạo file Web Log có phân bố thực tế tại ../data/web_logs_large.csv")

Đã tạo file Web Log có phân bố thực tế tại ../data/web_logs_large.csv


# Tạo dữ liệu bài 3

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# --- CẤU HÌNH ---
num_rows = 1000000
num_sensors = 100
sensor_ids = [f"S{i:03d}" for i in range(1, num_sensors + 1)]

# 1. Tạo "Hồ sơ sức khỏe" cho từng cảm biến
# 80 máy tốt, 15 máy cũ (nóng & rung hơn), 5 máy sắp hỏng (cực nóng & rung)
health_profiles = {}
for sid in sensor_ids:
    rand_val = np.random.rand()
    if rand_val < 0.80:
        health_profiles[sid] = {'base_temp': 45, 'base_vib': 15, 'volatility': 5} # Tốt
    elif rand_val < 0.95:
        health_profiles[sid] = {'base_temp': 65, 'base_vib': 30, 'volatility': 10} # Cũ
    else:
        health_profiles[sid] = {'base_temp': 75, 'base_vib': 45, 'volatility': 15} # Sắp hỏng

# 2. Sinh dữ liệu cơ bản
sensor_col = np.random.choice(sensor_ids, size=num_rows)
base_temps = np.array([health_profiles[s]['base_temp'] for s in sensor_col])
base_vibs = np.array([health_profiles[s]['base_vib'] for s in sensor_col])
volatilities = np.array([health_profiles[s]['volatility'] for s in sensor_col])

# Thêm nhiễu ngẫu nhiên (noise) dựa trên độ biến động của từng máy
temps = np.random.normal(base_temps, volatilities).round(1)
vibs = np.abs(np.random.normal(base_vibs, volatilities * 0.5)).round(2)
humidities = np.random.uniform(40, 80, size=num_rows).round(1)

# 3. Tạo "Sự cố bất ngờ" (Anomalies) - Chiếm khoảng 2% dữ liệu
anomaly_indices = np.random.choice(num_rows, size=int(num_rows * 0.02), replace=False)
temps[anomaly_indices] += np.random.uniform(20, 40, size=len(anomaly_indices)) # Đột ngột tăng vọt lên > 80
vibs[anomaly_indices] += np.random.uniform(20, 40, size=len(anomaly_indices))  # Đột ngột rung > 50

# 4. Gán trạng thái (Status) dựa trên logic thực tế
# Nếu nhiệt độ > 80 hoặc rung > 50 -> Critical. Nếu > 70 hoặc > 40 -> Warning. Còn lại Normal
conditions = [
    (temps > 80) | (vibs > 50),
    (temps > 70) | (vibs > 40)
]
choices = ['critical', 'warning']
statuses = np.select(conditions, choices, default='normal')

# 5. Đóng gói vào DataFrame và xuất file
data = {
    'sensor_id': sensor_col,
    'temperature': temps,
    'humidity': humidities,
    'vibration': vibs,
    'status': statuses
}

df_pandas = pd.DataFrame(data)

# Thêm Timestamp trong 30 ngày qua
start_date = datetime(2026, 4, 1)
df_pandas['timestamp'] = [start_date + timedelta(seconds=int(np.random.uniform(0, 30*24*3600))) 
                          for _ in range(num_rows)]

df_pandas = df_pandas.sort_values('timestamp').reset_index(drop=True)

os.makedirs('../data', exist_ok=True)
df_pandas.to_csv('../data/iot_sensors.csv', index=False)
print("Đã tạo file IoT Data với các sự cố thực tế tại ../data/iot_sensors_large.csv")

Đã tạo file IoT Data với các sự cố thực tế tại ../data/iot_sensors_large.csv
